## ADA - Reddit Analysis - Part3

## Preliminary Hypothesis

Let's consider a sub-reddit B being attacked and B'a control sub-reddit, similar to B before the attack but that didn't receive the attack.

|  Q  |We expect : | What would change our mind |
| --- | --- | --- |
| Q1| Increased nb of links toward B in the following days / weeks | B/B' close to 1 |
|  Q2 |  A higher ratio of negative links compared to control | A non changing negative ratio |
|   Q3| A higher ratio of negative links comming from B | A non - changing or lower ratio of negative outcoming links |
| Q4 | A temporary effect | The activity and negativity excess remains increased after 6 months |



### 0 : Data Import

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
BLUE, RED, GREY = "#1E67B5", "#D02F2F", "#7F7F7F"   # same colors as chapter 2
DAY = pd.Timedelta(days=1)                    # handy unit for time arithmetic

In [16]:
# Load the two TSV files (hyperlinks in post bodies and in post titles) and stack them.
data_dir = Path("Data")
columns = ["SOURCE_SUBREDDIT", "TARGET_SUBREDDIT", "POST_ID", "TIMESTAMP", "LINK_SENTIMENT"]

parts = []
for location in ["body", "title"]:
    part = pd.read_csv(data_dir / f"soc-redditHyperlinks-{location}.tsv", sep="\t", usecols=columns)
    part["LOCATION"] = location          # remember where the link was found
    parts.append(part)
links = pd.concat(parts, ignore_index=True)   # stack tables with the same columns

links["time"] = pd.to_datetime(links.pop("TIMESTAMP"))   # pop = take the column out, then convert it
links = links[links.SOURCE_SUBREDDIT != links.TARGET_SUBREDDIT]   # drop self-links

# Sort by time (ties broken by POST_ID/LOCATION so the order is deterministic).
# Sorting matters: "first negative link" and the fast time slicing below both rely on it.
links = links.sort_values(["time", "POST_ID", "LOCATION"]).reset_index(drop=True)

print(f"{len(links):,} links | {links.LINK_SENTIMENT.eq(-1).mean():.2%} negative")
print(links.time.min(), "->", links.time.max())

858,488 links | 9.58% negative
2013-12-31 16:20:20 -> 2017-04-30 16:58:21


### 1 - Events


In [17]:
MIN_LINKS = 5    # eligibility: at least 5 incoming links in the selection window

# For every subreddit, the time of the first negative link it ever received.
negatives = links[links.LINK_SENTIMENT == -1]
first_neg_time = negatives.groupby("TARGET_SUBREDDIT").time.min()

first_neg = negatives.drop_duplicates("TARGET_SUBREDDIT", keep="first")
events = first_neg.rename(columns={"SOURCE_SUBREDDIT": "A", "TARGET_SUBREDDIT": "B",
                                   "time": "t0", "LOCATION": "attack_location"})
events = events[["A", "B", "t0", "POST_ID", "attack_location"]]
n_attacked_ever = len(events)

# Keep only first attacks from 2015 onward (2014 is only used to check B was never attacked before).
events = events[events.t0 >= "2015-01-01"].reset_index(drop=True)
events["event_id"] = events.index
